In [0]:
# COMMAND ----------
 
# ─────────────────────────────────────────────────────────────────────
# 5. GOLD LAYER — KPIs for the business
#
# Step 1 KPIs (simple, no dimensions yet):
#   - Daily GMV (Gross Merchandise Value = sum of order_total)
#   - Daily order count
#   - Daily average order value
#   - By order status breakdown
#
# This is a full recompute for now.
# In Step 10 we switch to incremental aggregation.

In [0]:
from pyspark.sql import functions as F
silver_df=spark.table("file_based.ecommerce.silver_table")

gold_df=(
    silver_df.groupBy("order_date","order_status").agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum("order_total"),2).alias("gmv"),
        F.round(F.avg("order_total"),2).alias("avg_order"),
        F.sum("quantity").alias("total_units_sold")
    ).orderBy("order_date","order_status")
)

In [0]:
(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("file_based.ecommerce.gold_table")
)
 
print("Gold load complete.")
 
# COMMAND ----------
 
# ── Final check: Gold KPIs
display(spark.table("file_based.ecommerce.gold_table"))

In [0]:
bronze_count = spark.table("file_based.ecommerce.bronze_layer").count()
silver_count = spark.table("file_based.ecommerce.silver_table").count()
gold_count   = spark.table("file_based.ecommerce.gold_table").count()
 
print("=" * 45)
print("Pipeline Run Summary")
print("=" * 45)
print(f"  Bronze rows : {bronze_count}")
print(f"  Silver rows : {silver_count}")
print(f"  Gold rows   : {gold_count}")
print()
 
# Bronze and Silver should have the same count in Step 1
# (no dedup, no DQ filtering yet)
if bronze_count == silver_count:
    print("✓ Bronze == Silver row count  (expected in Step 1)")
else:
    print(f"✗ Mismatch: Bronze={bronze_count}, Silver={silver_count}")